# Tabular Q-Learning - Single Agent

## Transport Task in Grid World

Author: Mostafiz Rahman

Assignment: Part 1 - Tabular Q-Learning (Single Agent)

Objective: 
 Build a Q-Learning agent from scratch that learns to pick up an item from location A and deliver it to Location B in an nxn grid world.

## Problem Statement

The goal of this projects is to train a single agent using Tabular Q-Learning.

The agent must:

1. Start from a random position.
2. Find the pickup location (A).
3. Automatically pick up the item.
4. Deliver the item to the delivery location (B).
5. Learn the shortest path through training.

In [2]:
import random
import numpy as np

## Environment Setup

In this section, we create the grid world environment that will be used for training the Q-Learning agent.

In [3]:
GRID_SIZE = 5
has_package = False
# empty grid
grid = []

# create grid automatically
for i in range(GRID_SIZE):
    row = ["🟧"] * GRID_SIZE
    grid.append(row)
    
# print Grid
for row in grid:
    for value in row:
        print(value, end=" ")
    print()
    
# Robot Position added (Random)
robot_row = random.randint(0, GRID_SIZE -1)
robot_col = random.randint(0, GRID_SIZE -1)
    
# Pickup Position added (Random)
pickup_row = random.randint(0, GRID_SIZE -1)
pickup_col = random.randint(0, GRID_SIZE -1)
    
# Delivery Position added (Fixed)
delivery_row = GRID_SIZE -1
delivery_col = GRID_SIZE -1

# ==========>>>>>>>>> collision handling <<<<<<<<<<<<<================

# if Robot position and pickup position are same
while robot_row == pickup_row and robot_col == pickup_col:
    pickup_row = random.randint(0, GRID_SIZE - 1)
    pickup_col = random.randint(0, GRID_SIZE - 1)

# if Robot position and delivery position are same
while robot_row == delivery_row and robot_col == delivery_col:
    robot_row = random.randint(0, GRID_SIZE - 1)
    robot_col = random.randint(0, GRID_SIZE - 1)

# if pickup position and delivery position are same
while pickup_row == delivery_row and pickup_col == delivery_col:
    pickup_row = random.randint(0, GRID_SIZE - 1)
    pickup_col = random.randint(0, GRID_SIZE - 1)

# set the robot imoje
grid[robot_row][robot_col] = "🤖"
# set the pickup imoje
grid[pickup_row][pickup_col] = "📦"
# set the robot imoje
grid[delivery_row][delivery_col] = "🏠"
    
# grid print
for row in grid:
    for value in row:
        print(value, end=" ")
    print()

🟧 🟧 🟧 🟧 🟧 
🟧 🟧 🟧 🟧 🟧 
🟧 🟧 🟧 🟧 🟧 
🟧 🟧 🟧 🟧 🟧 
🟧 🟧 🟧 🟧 🟧 
🟧 🟧 🟧 🟧 🟧 
🟧 🟧 🟧 🟧 🟧 
🟧 🟧 🤖 🟧 🟧 
🟧 🟧 🟧 🟧 📦 
🟧 🟧 🟧 🟧 🏠 


## Agent Movement

In [4]:


while True:
    # Grid print
    for row in grid:
        for value in row:
           print(value, end=" ")
        print()
    
    # save previous position
    previous_row = robot_row
    previous_col = robot_col
    
    # User input for direction
    move = input("Enter direction: ").lower()
    print("Move: ", move)
    
    if move == "exit":
        print("Program End")
        break
    
    # remove robot for prev
    grid[robot_row][robot_col] = "🟧"
    
    # move robot
    if move == "left":
        robot_col -= 1
        
    elif move =="right":
        robot_col += 1
        
    elif move == "up":
        robot_row -= 1
        
    elif move == "down":
        robot_row += 1

    else:
        print("❌ Invalid Direction")
        
    # Boundary Check
    robot_row = max(0, min(robot_row, GRID_SIZE -1))
    robot_col = max(0, min(robot_col, GRID_SIZE -1))
    
    # Automatic Pickup
    if robot_row == pickup_row and robot_col == pickup_col:
       print("📦 Package Picked Up!")
       has_package = True
       
    # Automatic Delivery
    if has_package and robot_row == delivery_row and robot_col == delivery_col:
       print("🎉 Delivery Complete!")
       grid[robot_row][robot_col] = "🤖"
       for row in grid:
           for value in row:
               print(value, end=" ")
           print()
       break
    
    grid[robot_row][robot_col] = "🤖"
    if not has_package:
        grid[pickup_row][pickup_col] = "📦"
    grid[delivery_row][delivery_col] = "🏠"

🟧 🟧 🟧 🟧 🟧 
🟧 🟧 🟧 🟧 🟧 
🟧 🟧 🤖 🟧 🟧 
🟧 🟧 🟧 🟧 📦 
🟧 🟧 🟧 🟧 🏠 
Move:  exit
Program End


## Reward Structure

### Reward Design

The agent receives rewards and penalties based on its actions.

| Action | Reward |
|--------|--------|
| Pick up package | +10 |
| Deliver package | +100 |
| Normal movement | -1 |
| Invalid move / Hit boundary | -5 |

In [5]:
PICKUP_REWARD = 10
DELIVERY_REWARD = 100
STEP_PENALTY = -1
BOUNDARY_PENALTY = -5

## State Representation

In [6]:
# Current state

state = (
    (robot_row, robot_col),
    (pickup_row, pickup_col),
    has_package
)

print(state)

((2, 2), (3, 4), False)


## Q-Table
### What is a Q-Table?

A Q-Table stores the expected reward for taking each action from every possible state.

Rows = States

Columns = Actions

Values = Q-Values

In [7]:
# Action 
# Q-learing use this list

ACTIONS = [
    "up",
    "down",
    "left",
    "right",
    "up-left",
    "up-right",
    "down-left",
    "down-right",
]



## Initialize Q-Table

In [8]:
# Total Possible States
TOTAL_STATES = GRID_SIZE * GRID_SIZE * GRID_SIZE * GRID_SIZE * 2

# Create Q-Table
q_table = np.zeros((TOTAL_STATES, len(ACTIONS)))

print("Q-table shape: ", q_table.shape)

Q-table shape:  (1250, 8)


## Q-Learning Formula
Q(s,a) = Q(s,a) + α × [R + γ × max(Q(s',a')) − Q(s,a)]

### Q-Learning Formula Components

- **s** = Current State
- **a** = Current Action
- **R** = Reward
- **α (Alpha)** = Learning Rate
- **γ (Gamma)** = Discount Factor
- **s'** = Next State
- **max(Q(s',a'))** = Best Future Q-Value



### Calculate robot and pickup index

In [30]:
robot_index = robot_row * GRID_SIZE + robot_col
pickup_index = pickup_row * GRID_SIZE + pickup_col

state_index = robot_index * TOTAL_POSITIONS * 2 + pickup_index * 2 + int(has_package) # type: ignore

print(robot_index)
print(pickup_index)

12
19


## Action Selection

In [19]:
action = random.choice(ACTIONS)

print("selected Action: ", action)

selected Action:  down-right


## State Encoding Function

In [22]:
TOTAL_POSITIONS = GRID_SIZE * GRID_SIZE

def state_to_index(robot_row, robot_col, pickup_row, pickup_col, has_package):
    robot_index
    pickup_index
    
    state_index = (robot_index * TOTAL_POSITIONS * 2 + pickup_index * 2 + int(has_package))
    return state_index

In [23]:
state = state_to_index (
    robot_row, robot_col, pickup_row, pickup_col, has_package
)

print("State Index: ", state)

State Index:  638


## Epsilon-Greedy Action Selection

In [ ]:
EPSILON = 1.0
EPSILON_DECAY = 0.995
MIN_EPSILON = 0.01

if random.random() < EPSILON:
    # Exploration
    action = random.randint(0, len(ACTIONS) - 1)
    
else:
    # Exploitation
    action = np.argmax(q_table[state_index])

## Access Q-Table Useing State Index

In [31]:
state_index = state_to_index(
    robot_row,
    robot_col,
    pickup_row,
    pickup_col,
    has_package
)
print("Q-value: ", q_table[state_index])


Q-value:  [0. 0. 0. 0. 0. 0. 0. 0.]


## select best action

In [32]:
best_action = np.argmax(q_table[state_index])

print("best action index: ", best_action)
print ("best action: ", ACTIONS[best_action])

best action index:  0
best action:  up


In [33]:
numbers = [10, 50, 20, 90, 30]

out = np.argmax(numbers)
print(out)

3


Q-Learing Parameters

In [38]:
ALPHA = 0.1     # Learning Rate
GAMMA = 0.9     # Discount Factor
EPSILON = 1.0   # Exploration Rate 

EPSILON_DECAY = 0.995
MIN_EPSILON = 0.01

## Current Q-value

In [ ]:
state_index = state_to_index(
    robot_row,
    robot_col,
    pickup_row,
    pickup_col,
    has_package
)

# example aciton
action = 3

current_q = q_table[state_index][action]

print("state index: ", state_index)
print("Action Index: ", action)
print("current_q: ", current_q)


state index:  638
Action Index:  3
current_q:  0.0


## Maximum Future Q-Value

In [36]:
## next state
next_state_index = state_index

# best future Q-value
max_future_q = np.max(q_table[next_state_index])

print("Next state: ", next_state_index)
print("Maximun Future Q: ", max_future_q)

Next state:  638
Maximun Future Q:  0.0


## Update Q-value

In [40]:
reward = -1

new_q = current_q + ALPHA * (
    reward + GAMMA * max_future_q - current_q
)

print("Old Q: ", current_q)
print("New Q: ", new_q)

q_table[state_index][action] = new_q

print(q_table[state_index])

Old Q:  0.0
New Q:  -0.1
[ 0.   0.   0.  -0.1  0.   0.   0.   0. ]


## Traning Loop 

In [50]:
EPISODES = 20000

for episode in range(EPISODES):
    
    # Restart Package Status
    has_package = False
    
    # create empty grid
    grid = []
    
    for i in range(GRID_SIZE):
        row = ["🟧"] * GRID_SIZE
        grid.append(row)
    
    # Random Robot Position
    robot_row = random.randint(0, GRID_SIZE - 1)
    robot_col = random.randint(0, GRID_SIZE - 1)
    
    # Random Pickup possition
    pickup_row = random.randint(0, GRID_SIZE - 1)
    pickup_col = random.randint(0, GRID_SIZE - 1)
    
    # Delivery possition
    delivery_row = GRID_SIZE -1
    delivery_col = GRID_SIZE -1
    
    # colision handling 
     
    while robot_row == pickup_row and robot_col == pickup_col:
        pickup_row = random.randint(0, GRID_SIZE - 1)
        pickup_col = random.randint(0, GRID_SIZE - 1)

    while robot_row == delivery_row and robot_col == delivery_col:
        robot_row = random.randint(0, GRID_SIZE - 1)
        robot_col = random.randint(0, GRID_SIZE - 1)

    while pickup_row == delivery_row and pickup_col == delivery_col:
        pickup_row = random.randint(0, GRID_SIZE - 1)
        pickup_col = random.randint(0, GRID_SIZE - 1)
        
        
    # Current State
    state_index = state_to_index(
        robot_row,
        robot_col,
        pickup_row,
        pickup_col,
        has_package
    )

    # Epsilon-Greedy Action Selection
    if random.random() < EPSILON:
        action = random.randint(0, len(ACTIONS) - 1)
    else:
        action = np.argmax(q_table[state_index])
        
        # Save Current Position
    previous_row = robot_row
    previous_col = robot_col

    # Execute Action
    if action == 0:          # Up
        robot_row -= 1

    elif action == 1:        # Down
        robot_row += 1

    elif action == 2:        # Left
        robot_col -= 1

    elif action == 3:        # Right
        robot_col += 1

    elif action == 4:        # Up Left
        robot_row -= 1
        robot_col -= 1

    elif action == 5:        # Up Right
        robot_row -= 1
        robot_col += 1

    elif action == 6:        # Down Left
        robot_row += 1
        robot_col -= 1

    elif action == 7:        # Down Right
        robot_row += 1
        robot_col += 1
    
    q_table[state_index]
    
    ## Boundary Check
    
    # Keep Robot Inside Grid
    robot_row = max(0, min(robot_row, GRID_SIZE - 1))
    robot_col = max(0, min(robot_col, GRID_SIZE - 1))

## Execute Action